In [1]:
# Built-ins
import os
from pathlib import Path
!pip install hmmlearn
# Data science / utilities
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mode
# Scikit-learn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# HMM
from hmmlearn.hmm import GaussianHMM

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, TimeDistributed, GlobalAveragePooling2D, LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 575.7 kB/s eta 0:00:00


2026-04-07 12:18:27.768056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775564307.999617      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775564308.065765      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775564308.602598      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775564308.602654      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775564308.602657      17 computation_placer.cc:177] computation placer alr

In [2]:
# Define the path to the dataset
base_path = '/kaggle/input/datasets/haidang2519/deepmfake/ff23/ff23'
categories = ['fake', 'real']

# Initialize a list to hold data
data = []

# Process each category
for category in categories:
    category_path = os.path.join(base_path, category)
    for filename in os.listdir(category_path):
        if filename.endswith('.jpg'):
            try:
                id_part, frame_part = filename.split('_frame_')
                id_ = id_part.split('_')[0]
                frame = frame_part.split('.')[0]
                data.append({
                    'filename': filename,
                    'path': os.path.join(category_path, filename),
                    'id': int(id_),
                    'frame': int(frame),
                    'label': category
                })
            except ValueError:
                continue

# Convert the data to a DataFrame
df = pd.DataFrame(data)
df['label_id'] = df['label'].map({'fake': 0, 'real': 1})

In [3]:
df['video_key'] = df['id'].astype(str) + "_" + df['label']

from collections import defaultdict

video_dict = defaultdict(list)
labels = {}

for _, row in df.iterrows():
    key = row['video_key']
    video_dict[key].append(row['path'])
    labels[key] = row['label_id']


In [4]:
# Chuẩn bị dữ liệu
video_keys = list(video_dict.keys())
video_labels = [labels[k] for k in video_keys]

# Cấu hình
img_size = (224, 224)
batch_size = 16
epochs = 50
n_splits = 5
sequence_len = 10
results = []
all_histories = []

# Custom data generator để nạp chuỗi ảnh từ video_dict
class VideoSequence(tf.keras.utils.Sequence):
    def __init__(self, video_keys, video_dict, labels, batch_size, img_size, sequence_len=10, augment=False):
        self.video_keys = video_keys
        self.video_dict = video_dict
        self.labels = labels
        self.batch_size = batch_size
        self.img_size = img_size
        self.sequence_len = sequence_len
        self.augment = augment
        self.datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=15 if augment else 0,
            zoom_range=0.1 if augment else 0,
            horizontal_flip=augment
        )

    def __len__(self):
        return int(np.ceil(len(self.video_keys) / self.batch_size))

    def __getitem__(self, idx):
        batch_keys = self.video_keys[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_X, batch_y = [], []

        for key in batch_keys:
            frames = video_dict[key][:self.sequence_len]
            imgs = []
            for path in frames:
                img = cv2.imread(path)
                img = cv2.resize(img, self.img_size)
                img = self.datagen.random_transform(img) if self.augment else img
                img = img.astype('float32') / 255.0
                imgs.append(img)
            while len(imgs) < self.sequence_len:
                imgs.append(np.zeros((*self.img_size, 3), dtype='float32'))  # padding
            batch_X.append(imgs)
            batch_y.append(self.labels[key])

        return np.array(batch_X), np.array(batch_y)

# Hàm xây dựng mô hình
def build_model(sequence_len, img_size):
    base_model = MobileNetV2(input_shape=(*img_size, 3), include_top=False, weights='imagenet')

    # Freeze toàn bộ backbone để giảm overfit
    base_model.trainable = False

    # CNN feature extractor
    model_out = GlobalAveragePooling2D()(base_model.output)
    model = Model(inputs=base_model.input, outputs=model_out)

    # Sequence input
    input_seq = Input(shape=(sequence_len, *img_size, 3))
    x = TimeDistributed(model)(input_seq)

    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.5)(x)

    # Không dùng nhiều Dense, chỉ một đầu ra
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=input_seq, outputs=output)
    return model

# K-Fold huấn luyện
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(video_keys, video_labels), 1):
    print(f"\n===== Fold {fold} =====")

    trainval_keys = [video_keys[i] for i in trainval_idx]
    test_keys = [video_keys[i] for i in test_idx]

    y_trainval = [labels[k] for k in trainval_keys]
    train_keys, val_keys = train_test_split(trainval_keys, test_size=0.1, stratify=y_trainval, random_state=fold)

    train_gen = VideoSequence(train_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=True)
    val_gen   = VideoSequence(val_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=False)
    test_gen  = VideoSequence(test_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=False)

    model = build_model(sequence_len, img_size)
    model.compile(optimizer=Adamax(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    model_path = f"best_model_fold{fold}.h5"
    checkpoint = ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    earlystop = EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=epochs, callbacks=[checkpoint, earlystop, reduce_lr], verbose=1)

    all_histories.append(history.history)

    model.load_weights(model_path)

    y_true = [labels[k] for k in test_keys]
    y_pred_prob = model.predict(test_gen).ravel()
    y_pred = (y_pred_prob > 0.5).astype(int)

    results.append({
        'fold': fold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_pred_prob)
    })

# Tổng kết kết quả
print("\n📊 Tổng kết kết quả các fold:")
for r in results:
    print(f"Fold {r['fold']}: Accuracy={r['accuracy']:.4f}, F1={r['f1']:.4f}, AUC={r['auc']:.4f}")



===== Fold 1 =====


2026-04-07 12:18:56.089617: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4858 - loss: 0.7429
Epoch 1: val_accuracy improved from -inf to 0.53750, saving model to best_model_fold1.h5


...

90/90 ━━━━━━━━━━━━━━━━━━━━ 317s 3s/step - accuracy: 0.5298 - loss: 0.7354 - val_accuracy: 0.5750 - val_loss: 0.6731 - learning_rate: 1.0000e-04
Epoch 2/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5117 - loss: 0.7108
Epoch 2: val_accuracy improved from 0.57500 to 0.61875, saving model to best_model_fold5.h5


90/90 ━━━━━━━━━━━━━━━━━━━━ 267s 3s/step - accuracy: 0.5123 - loss: 0.7105 - val_accuracy: 0.6187 - val_loss: 0.6496 - learning_rate: 1.0000e-04
Epoch 3/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5945 - loss: 0.6653
Epoch 3: val_accuracy improved from 0.61875 to 0.65000, saving model to best_model_fold5.h5


90/90 ━━━━━━━━━━━━━━━━━━━━ 268s 3s/step - accuracy: 0.5946 - loss: 0.6653 - val_accuracy: 0.6500 - val_loss: 0.6370 - learning_rate: 1.0000e-04
Epoch 4/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6170 - loss: 0.6514
Epoch 4: val_accuracy improved from 0.65000 to 0.70000, saving model to best_model_fold5.h5


90/90 ━━━━━━━━━━━━━━━━━━━━ 267s 3s/step - accuracy: 0.6171 - loss: 0.6513 - val_accuracy: 0.7000 - val_loss: 0.6264 - learning_rate: 1.0000e-04
Epoch 5/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6272 - loss: 0.6397
Epoch 5: val_accuracy did not improve from 0.70000
90/90 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.6273 - loss: 0.6396 - val_accuracy: 0.6812 - val_loss: 0.6163 - learning_rate: 1.0000e-04
Epoch 6/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6577 - loss: 0.6159
Epoch 6: val_accuracy improved from 0.70000 to 0.71875, saving model to best_model_fold5.h5


90/90 ━━━━━━━━━━━━━━━━━━━━ 317s 3s/step - accuracy: 0.6577 - loss: 0.6159 - val_accuracy: 0.7188 - val_loss: 0.6027 - learning_rate: 1.0000e-04
Epoch 7/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6564 - loss: 0.6098
Epoch 7: val_accuracy did not improve from 0.71875
90/90 ━━━━━━━━━━━━━━━━━━━━ 262s 3s/step - accuracy: 0.6565 - loss: 0.6097 - val_accuracy: 0.7063 - val_loss: 0.5939 - learning_rate: 1.0000e-04
Epoch 8/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6714 - loss: 0.5755
Epoch 8: val_accuracy did not improve from 0.71875
90/90 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.6715 - loss: 0.5755 - val_accuracy: 0.6812 - val_loss: 0.5865 - learning_rate: 1.0000e-04
Epoch 9/50
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7162 - loss: 0.5739
Epoch 9: val_accuracy did not improve from 0.71875
90/90 ━━━━━━━━━━━━━━━━━━━━ 265s 3s/step - accuracy: 0.7163 - loss: 0.5738 - val_accuracy: 0.7188 - val_loss: 0.5774 - learning_rate: 1.0000e-04
Epoch 10/50
90/90 ━━━━

In [5]:
import pandas as pd

# Giả sử results đã có và bạn đã tạo results_df
results_df = pd.DataFrame(results)

# Tính các chỉ số
accuracy_mean = results_df['accuracy'].mean()
accuracy_std = results_df['accuracy'].std()  # dùng sample std (chia cho n-1)
accuracy_range = results_df['accuracy'].max() - results_df['accuracy'].min()
accuracy_cv_percent = (accuracy_std / accuracy_mean) * 100

# In kết quả
print("📊 Kết quả trung bình:")
print(results_df.mean(numeric_only=True))

print(f"\n✅ CV Accuracy (Mean Accuracy): {accuracy_mean:.4f}")
print(f"📈 Range Accuracy: {accuracy_range:.4f}")
print(f"📉 Accuracy CV% (std/mean): {accuracy_cv_percent:.2f}%")

# Hiển thị bảng kết quả nếu cần
results_df


📊 Kết quả trung bình:
fold         3.000000
accuracy     0.714000
precision    0.727606
recall       0.685000
f1           0.704837
auc          0.788560
dtype: float64

✅ CV Accuracy (Mean Accuracy): 0.7140
📈 Range Accuracy: 0.0350
📉 Accuracy CV% (std/mean): 1.81%


,fold,accuracy,precision,recall,f1,auc
0,1,0.720,0.739130,0.680,0.708333,0.814425
1,2,0.730,0.730000,0.730,0.730000,0.798300
2,3,0.710,0.710000,0.710,0.710000,0.761350
3,4,0.715,0.723958,0.695,0.709184,0.781600
4,5,0.695,0.734940,0.610,0.666667,0.787125
